# 01 — M0 Independent Heads

담당자는 이 노트북만 실행하면 됩니다.

- Encoder: `monologg/koelectra-small-v3-discriminator`
- Model: **KoELECTRA-small + Independent Item Heads**
- Seeds: `42, 43, 44`
- 목적: Item Query Decoder와 비교하기 위한 구조 baseline
- 데이터: `data/structure_b_train_valid.jsonl` / `data/structure_b_test.jsonl`


## 1. 공개 저장소 clone 및 패키지 설치

In [ ]:
import os, subprocess
from pathlib import Path

repo_dir = Path('/content/pumpkin_public')
if not repo_dir.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/hammonsterwow/pumpkin_public.git',
        str(repo_dir),
    ], check=True)

os.chdir(repo_dir)
subprocess.run([
    'pip', 'install', '-q', '-r',
    'experiments/nlu/kips_2026_item_query/requirements-colab.txt',
], check=True)
print('repo:', repo_dir)


## 2. GPU 및 고정 설정 확인

In [ ]:
import torch
from experiments.nlu.kips_2026_item_query.config import MAIN_CONFIGS, MAIN_SEEDS, MODEL_NAME
config = MAIN_CONFIGS[0]
assert config.name == 'M0_independent_heads'
print('experiment:', config)
print('seeds:', MAIN_SEEDS)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))


## 3. 동일 데이터 split 준비

In [ ]:
from pathlib import Path
from transformers import AutoTokenizer
from experiments.nlu.kips_2026_item_query.run_all import canonical_paths, prepare_data, save_manifest

train_valid_path, test_path = canonical_paths()
pool, train_records, valid_records, test_records = prepare_data(train_valid_path, test_path)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
result_root = Path('/content/kips_M0_results')
result_root.mkdir(parents=True, exist_ok=True)
tokenizer.save_pretrained(result_root / 'tokenizer')
save_manifest(result_root, train_valid_path, test_path, pool, train_records, valid_records, test_records)


## 4. M0 × seeds 42/43/44 — 총 3회 학습 및 최종 test

In [ ]:
from experiments.nlu.kips_2026_item_query.train_eval import run_suite, summarize, make_paper_table
runs = run_suite([config], MAIN_SEEDS, train_records, valid_records, test_records, tokenizer, result_root)
runs.to_csv(result_root / 'M0_runs.csv', index=False)
summary = summarize(runs)
summary.to_csv(result_root / 'M0_summary.csv', index=False)
paper_table = make_paper_table(summary)
paper_table.to_csv(result_root / 'M0_paper_table.csv', index=False)
display(runs)
display(paper_table)


## 5. 결과 ZIP 다운로드

In [ ]:
import shutil
from google.colab import files
zip_path = shutil.make_archive('/content/kips_M0_results', 'zip', result_root)
files.download(zip_path)
